In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PageRank").getOrCreate()
sc = spark.sparkContext

In [3]:
def load_graph(file):
    lines = sc.textFile(file)

    edges = lines.map(lambda x: tuple(map(int, x.split()))) \
                 .distinct()

    return edges

edges = load_graph("whole.txt")

In [4]:
links = edges.groupByKey().mapValues(list).cache()

nodes = edges.flatMap(lambda x: x).distinct()
N = nodes.count()

print("Total nodes:", N)

Total nodes: 1000


In [5]:
beta = 0.8
iterations = 40

ranks = nodes.map(lambda x: (x, 1.0 / N))

In [6]:
for _ in range(iterations):

    contribs = links.join(ranks).flatMap(
        lambda x: [(neighbor, x[1][1] / len(x[1][0])) for neighbor in x[1][0]]
    )

    ranks = contribs.reduceByKey(lambda a, b: a + b) \
                    .mapValues(lambda rank: (1 - beta)/N + beta * rank)

In [7]:
result = ranks.collect()

sorted_nodes = sorted(result, key=lambda x: x[1], reverse=True)

print("Top 5 nodes:")
for node, score in sorted_nodes[:5]:
    print(node, score)

print("\nBottom 5 nodes:")
for node, score in sorted_nodes[-5:]:
    print(node, score)

Top 5 nodes:
263 0.002020291181518219
537 0.0019433415714531492
965 0.0019254478071662631
243 0.001852634016241731
285 0.0018273721700645144

Bottom 5 nodes:
408 0.00038779848719291705
424 0.00035481538649301454
62 0.00035314810510596274
93 0.0003513568937516577
558 0.0003286018525215297


In [8]:
print("Sum of ranks:", sum([x[1] for x in result]))

Sum of ranks: 0.9999999999999999
